## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "llama3.2:latest"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [5]:
retriever = vectorstore.as_retriever()

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='5b891a70-88c1-4dd2-864e-0c602e210f06', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content="There are several notable individuals named Avery, so it's possible that you're referring to one of the following:\n\n1. Avery Williamson: An American football linebacker who currently plays for the New York Jets.\n2. Avery Bradley: An American professional basketball player who played in the NBA from 2010 to 2020.\n3. Avery Jones: A British singer-songwriter and musician.\n4. Avery Wines: An American winery based in Sonoma County, California.\n\nIf none of these individuals match the person you're thinking of, could you please provide more context or information about who Avery is?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-07-01T12:57:43.7015205Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23805303600, 'load_duration': 7505400200, 'prompt_eval_count': 29, 'prompt_eval_duration': 838140000, 'eval_count': 122, 'eval_duration': 15434468000, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--0e41804e

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("Who is Averi Lancaster?", [])

"Avery Lancaster is actually Emily Carter's colleague and not a public figure I have information on. However, based on the context provided, it seems that Avery Lancaster is likely an employee or colleague of Emily Carter at Insurellm.\n\nIf you're looking for more information about Emily Carter, she is a talented individual who exemplifies the kind of talent that drives Insurellm's success and is an invaluable asset to the company."

In [11]:
gr.ChatInterface(answer_question).launch()

c:\Users\DELL\Desktop\AI-Engineer-core-LLM\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
